In [ ]:
import os, sys, json, argparse
os.system('wget -O evals/all_images.pt https://huggingface.co/datasets/pscotti/mindeyev2/resolve/main/evals/all_images.pt')
os.system('wget -O evals/all_git_generated_captions.pt https://huggingface.co/datasets/pscotti/mindeyev2/resolve/main/evals/all_git_generated_captions.pt')
os.system('wget -O evals/all_captions.pt https://huggingface.co/datasets/pscotti/mindeyev2/resolve/main/evals/all_captions.pt')

import numpy as np, math
from einops import rearrange
import time, random, string, h5py
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import types
from torchvision import transforms
from accelerate import Accelerator   
import transformers
from transformers import pipeline, AutoTokenizer

sys.path.append('generative_models/')
import sgm
from generative_models.sgm.modules.encoders.modules import (
    FrozenOpenCLIPImageEmbedder, FrozenCLIPEmbedder, FrozenOpenCLIPEmbedder2
)
from generative_models.sgm.models.diffusion import DiffusionEngine
from generative_models.sgm.util import append_dims
from omegaconf import OmegaConf

torch.backends.cuda.matmul.allow_tf32 = True

import utils
from models import *

accelerator = Accelerator(split_batches=False, mixed_precision="fp16")
device = accelerator.device


device_0 = torch.device("cuda:0")   # text embedders, clip_img_embedder
device_1 = torch.device("cuda:1")   # base_engine (SDXL), latents


# PATCH 1 — mock xformers with native Pytorch attention
if "xformers" not in sys.modules:
    mock_xformers = types.ModuleType("xformers")
    mock_ops = types.ModuleType("xformers.ops")
    mock_xformers.ops = mock_ops
    sys.modules["xformers"] = mock_xformers
    sys.modules["xformers.ops"] = mock_ops

    def native_sdpa(query, key, value, *args, **kwargs):
        needs_transpose = query.dim() == 4
        if needs_transpose:
            query, key, value = query.transpose(1,2), key.transpose(1,2), value.transpose(1,2)
        out = F.scaled_dot_product_attention(query, key, value)
        if needs_transpose:
            out = out.transpose(1, 2)
        return out

    mock_ops.memory_efficient_attention = native_sdpa

import xformers
print("device:", device)

LOCAL RANK  0
device: cuda


In [ ]:
if utils.is_interactive():
    model_name = "final_subj01_pretrained_40sess_24bs"
    print("model_name:", model_name)

    path = "datasets"   

    jupyter_args = f"--model_name={model_name} --subj=1 --cache_dir={path}"
    print(jupyter_args)
    jupyter_args = jupyter_args.split()

    from IPython.display import clear_output
    %load_ext autoreload
    %autoreload 2

model_name: final_subj01_pretrained_40sess_24bs
--model_name=final_subj01_pretrained_40sess_24bs --subj=1


In [ ]:
parser = argparse.ArgumentParser(description="Model Training Configuration")
parser.add_argument("--model_name", type=str, default="testing")
parser.add_argument("--subj", type=int, default=1, choices=[1,2,3,4,5,6,7,8])
parser.add_argument("--seed", type=int, default=42)
parser.add_argument(
    "--cache_dir", type=str, default=os.getcwd(),
    help="Folder with downloaded checkpoints (e.g., zavychromaxl_v30.safetensors)",
)

if utils.is_interactive():
    args = parser.parse_args(jupyter_args)
else:
    args = parser.parse_args()

for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name)

utils.seed_everything(seed)
os.makedirs("evals", exist_ok=True)
os.makedirs(f"evals/{model_name}", exist_ok=True)



all_images      = torch.load("evals/all_images.pt")
all_recons      = torch.load(f"evals/{model_name}/{model_name}_all_recons.pt")
all_clipvoxels  = torch.load(f"evals/{model_name}/{model_name}_all_clipvoxels.pt")
all_blurryrecons= torch.load(f"evals/{model_name}/{model_name}_all_blurryrecons.pt")
all_predcaptions= torch.load(f"evals/{model_name}/{model_name}_all_predcaptions.pt")



# Resize to 768×768 for SDXL
all_recons       = transforms.Resize((768, 768))(all_recons).float()
all_blurryrecons = transforms.Resize((768, 768))(all_blurryrecons).float()

print(model_name)
print(all_images.shape, all_recons.shape, all_clipvoxels.shape,
      all_blurryrecons.shape, len(all_predcaptions))

/admin/home-paulscotti/mindeye/lib/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


final_subj01_pretrained_40sess_24bs
torch.Size([1000, 3, 224, 224]) torch.Size([1000, 3, 768, 768]) torch.Size([1000, 256, 1664]) torch.Size([1000, 3, 768, 768]) (1000,)


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

# unclip6
config = OmegaConf.load("generative_models/configs/unclip6.yaml")
config = OmegaConf.to_container(config, resolve=True)
sampler_config = config["model"]["params"]["sampler_config"]
sampler_config['params']['num_steps'] = 38

# sdxl
config = OmegaConf.load("generative_models/configs/inference/sd_xl_base.yaml")
config = OmegaConf.to_container(config, resolve=True)
refiner_params = config["model"]["params"]

network_config            = refiner_params["network_config"]
denoiser_config           = refiner_params["denoiser_config"]
first_stage_config        = refiner_params["first_stage_config"]
conditioner_config        = refiner_params["conditioner_config"]
scale_factor              = refiner_params["scale_factor"]
disable_first_stage_autocast = refiner_params["disable_first_stage_autocast"]


base_ckpt_path = f'{cache_dir}/zavychromaxl_v30.safetensors'  

base_engine = DiffusionEngine(
    network_config=network_config,
    denoiser_config=denoiser_config,
    first_stage_config=first_stage_config,
    conditioner_config=conditioner_config,
    sampler_config=sampler_config,
    scale_factor=scale_factor,
    disable_first_stage_autocast=disable_first_stage_autocast,
    ckpt_path=base_ckpt_path,
)
base_engine.eval().requires_grad_(False)

# PATCH 2 — safe decode for base_engine
if hasattr(base_engine, 'first_stage_model'):
    base_engine.first_stage_model.to(device=device_1, dtype=torch.float32)

_orig_decode = base_engine.decode_first_stage


def safe_decode_base(z, *args, **kwargs):
    # Force to float16
    return _orig_decode(z.to(device_1, dtype=torch.float16), *args, **kwargs)

base_engine.decode_first_stage = safe_decode_base

# PATCH 3 — move all SGM tensors on device_1
for module in base_engine.modules():
    for attr_name in dir(module):
        if attr_name.startswith('__'):
            continue
        try:
            attr_val = getattr(module, attr_name)
            if isinstance(attr_val, torch.Tensor):
                setattr(module, attr_name, attr_val.to(device_1))
        except Exception:
            pass

# base_engine -> GPU 1
base_engine.to(device_1, dtype=torch.float16)

# Text embedders -> GPU 0 
base_text_embedder1 = FrozenCLIPEmbedder(
    layer=conditioner_config['params']['emb_models'][0]['params']['layer'],
    layer_idx=conditioner_config['params']['emb_models'][0]['params']['layer_idx'],
)
base_text_embedder1.to(device_0)

base_text_embedder2 = FrozenOpenCLIPEmbedder2(
    arch=conditioner_config['params']['emb_models'][1]['params']['arch'],
    version=conditioner_config['params']['emb_models'][1]['params']['version'],
    freeze=conditioner_config['params']['emb_models'][1]['params']['freeze'],
    layer=conditioner_config['params']['emb_models'][1]['params']['layer'],
    always_return_pooled=conditioner_config['params']['emb_models'][1]['params']['always_return_pooled'],
    legacy=conditioner_config['params']['emb_models'][1]['params']['legacy'],
)
base_text_embedder2.to(device_0)

# Conditional/unconditional embeddings
batch = {
    "txt": "",
    "original_size_as_tuple": torch.ones(1, 2).to(device_0) * 768,
    "crop_coords_top_left":   torch.zeros(1, 2).to(device_0),
    "target_size_as_tuple":   torch.ones(1, 2).to(device_0) * 1024,
}


base_engine_cpu_conditioner = base_engine.conditioner.to(device_0)
out = base_engine_cpu_conditioner(batch)

crossattn      = out["crossattn"].to(device_1)
vector_suffix  = out["vector"][:, -1536:].to(device_1)
print("crossattn", crossattn.shape)
print("vector_suffix", vector_suffix.shape)
print("---")

batch_uc = {
    "txt": ("painting, extra fingers, mutated hands, poorly drawn hands, "
            "poorly drawn face, deformed, ugly, blurry, bad anatomy, bad proportions, "
            "extra limbs, cloned face, skinny, glitchy, double torso, extra arms, "
            "extra hands, mangled fingers, missing lips, ugly face, distorted face, "
            "extra legs, anime"),
    "original_size_as_tuple": torch.ones(1, 2).to(device_0) * 768,
    "crop_coords_top_left":   torch.zeros(1, 2).to(device_0),
    "target_size_as_tuple":   torch.ones(1, 2).to(device_0) * 1024,
}
out = base_engine_cpu_conditioner(batch_uc)
crossattn_uc = out["crossattn"].to(device_1)
vector_uc    = out["vector"].to(device_1)
print("crossattn_uc", crossattn_uc.shape)
print("vector_uc", vector_uc.shape)

# Move conditionrer on gpu1 after using it
base_engine.conditioner.to(device_1)

SpatialTransformer: Found context dims [2048] of depth 1, which does not match the specified 'depth' of 2. Setting context_dim to [2048, 2048] now.
SpatialTransformer: Found context dims [2048] of depth 1, which does not match the specified 'depth' of 2. Setting context_dim to [2048, 2048] now.
SpatialTransformer: Found context dims [2048] of depth 1, which does not match the specified 'depth' of 10. Setting context_dim to [2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048] now.
SpatialTransformer: Found context dims [2048] of depth 1, which does not match the specified 'depth' of 10. Setting context_dim to [2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048] now.
SpatialTransformer: Found context dims [2048] of depth 1, which does not match the specified 'depth' of 10. Setting context_dim to [2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048] now.
SpatialTransformer: Found context dims [2048] of depth 1, which does not match the specified 'depth' of 10. Setti

Initialized embedder #0: FrozenCLIPEmbedder with 123060480 params. Trainable: False
Initialized embedder #1: FrozenOpenCLIPEmbedder2 with 694659841 params. Trainable: False
Initialized embedder #2: ConcatTimestepEmbedderND with 0 params. Trainable: False
Initialized embedder #3: ConcatTimestepEmbedderND with 0 params. Trainable: False
Initialized embedder #4: ConcatTimestepEmbedderND with 0 params. Trainable: False
Restored from /weka/proj-fmri/paulscotti/stable-research/zavychromaxl_v30.safetensors with 1 missing and 1 unexpected keys
Missing Keys: ['denoiser.sigmas']
Unexpected Keys: ['conditioner.embedders.0.transformer.text_model.embeddings.position_ids']
crossattn torch.Size([1, 77, 2048])
vector_suffix torch.Size([1, 1536])
---
crossattn_uc torch.Size([1, 77, 2048])
vector_uc torch.Size([1, 2816])


In [ ]:
if utils.is_interactive():
    plotting = True

num_samples         = 1
img2img_timepoint   = 13   
base_engine.sampler.guider.scale = 5  # guidance scale

def denoiser(x, sigma, c):
    return base_engine.denoiser(base_engine.model, x, sigma, c)


if utils.is_interactive() or num_samples > 1:
    clip_img_embedder = FrozenOpenCLIPImageEmbedder(
        arch="ViT-bigG-14",
        version="laion2b_s39b_b160k",
        output_tokens=True,
        only_tokens=True,
    )
    clip_img_embedder.to(device_0)   

In [ ]:
import os
from torchvision import transforms
import matplotlib.pyplot as plt

all_enhancedrecons = None


plotting = False 


png_dir = f"evals/{model_name}/enhanced_pngs"
os.makedirs(png_dir, exist_ok=True)

original_discretization = base_engine.sampler.discretization
def patched_discretization(*args, **kwargs):
    return original_discretization(*args, **kwargs).to(device_1)
base_engine.sampler.discretization = patched_discretization

for img_idx in tqdm(range(len(all_recons))):
# for img_idx in tqdm(range(0, 10)): # uncomment if you want to run the code just for 10 images

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
        with base_engine.ema_scope():
            base_engine.sampler.num_steps = 25

            image = all_recons[[img_idx]]

            if plotting:
                print("blur pixcorr:", utils.pixcorr(
                    all_blurryrecons[[img_idx]].float(), all_images[[img_idx]].float()))
                print("blur cossim:", nn.functional.cosine_similarity(
                    clip_img_embedder(utils.resize(all_blurryrecons[[img_idx]].float(), 256).to(device_0)).flatten(1),
                    clip_img_embedder(utils.resize(all_images[[img_idx]].float(), 224).to(device_0)).flatten(1)))
                print("recon pixcorr:", utils.pixcorr(image, all_images[[img_idx]].float()))
                print("recon cossim:", nn.functional.cosine_similarity(
                    clip_img_embedder(utils.resize(image, 224).to(device_0)).flatten(1),
                    clip_img_embedder(utils.resize(all_images[[img_idx]].float(), 224).to(device_0)).flatten(1)))

            
            image = image.to(device_1, dtype=torch.float16) 
            prompt = all_predcaptions[img_idx]
            if isinstance(prompt, (list, np.ndarray)):
                prompt = str(prompt[0])

            if plotting:
                print("prompt:", prompt)
                plt.imshow(transforms.ToPILImage()(all_blurryrecons[img_idx].float())); plt.show()
                plt.imshow(transforms.ToPILImage()(all_recons[img_idx].float()));       plt.show()
                plt.imshow(transforms.ToPILImage()(image[0].cpu()));                    plt.show()

            # Encoding image in latent space
            assert image.shape[-1] == 768
            z = base_engine.encode_first_stage(image * 2 - 1).repeat(num_samples, 1, 1, 1)

            # Text embeddings 
            openai_clip_text               = base_text_embedder1(prompt)          # GPU 0
            clip_text_tokenized, clip_text_emb = base_text_embedder2(prompt)      # GPU 0
            clip_text_emb        = torch.hstack((clip_text_emb, vector_suffix.to(device_0)))
            clip_text_tokenized  = torch.cat((openai_clip_text, clip_text_tokenized), dim=-1)

            
            c = {
                "crossattn": clip_text_tokenized.to(device_1).repeat(num_samples, 1, 1),
                "vector":    clip_text_emb.to(device_1).repeat(num_samples, 1),
            }
            uc = {
                "crossattn": crossattn_uc.to(device_1).repeat(num_samples, 1, 1),
                "vector":    vector_uc.to(device_1).repeat(num_samples, 1),
            }


            # Img2img with SDXL 
            noise  = torch.randn_like(z)
            sigmas = base_engine.sampler.discretization(
                base_engine.sampler.num_steps).to(device_1)
            init_z = (z + noise * append_dims(sigmas[-img2img_timepoint], z.ndim)) \
                     / torch.sqrt(1.0 + sigmas[0] ** 2.0)
            sigmas = sigmas[-img2img_timepoint:].repeat(num_samples, 1)

            base_engine.sampler.num_steps = sigmas.shape[-1] - 1
            noised_z, _, _, _, c, uc = base_engine.sampler.prepare_sampling_loop(
                init_z, cond=c, uc=uc, num_steps=base_engine.sampler.num_steps)

            for timestep in range(base_engine.sampler.num_steps):
                noised_z = base_engine.sampler.sampler_step(
                    sigmas[:, timestep], sigmas[:, timestep + 1],
                    denoiser, noised_z, cond=c, uc=uc, gamma=0)

            samples_z_base = noised_z
            samples_x      = base_engine.decode_first_stage(samples_z_base)   # usa safe_decode
            samples        = torch.clamp((samples_x + 1.0) / 2.0, min=0.0, max=1.0)

            # Choosing best sample
            if num_samples == 1:
                samples = samples[0]
            else:
                sample_cossim = nn.functional.cosine_similarity(
                    clip_img_embedder(utils.resize(samples, 224).to(device_0)).flatten(1),
                    clip_img_embedder(utils.resize(all_images[[img_idx]].float(), 224).to(device_0)).flatten(1))
                which_sample = torch.argmax(sample_cossim)

                if plotting:
                    for n in range(num_samples):
                        plt.imshow(transforms.ToPILImage()(samples[n]))
                        plt.show()
                        if (n == which_sample).item(): print("CHOSEN ABOVE")
                    raise StopIteration  # interrompe con plotting=True

                samples = samples[which_sample]

            # Converting single image tensor
            final_pil_img = transforms.ToPILImage()(samples.cpu())
            final_pil_img.save(f"{png_dir}/enhanced_img_{img_idx}.png")

            # Saving combined image: original, blurred, reconstructed, enhanced
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            fig.suptitle(str(prompt).capitalize(), fontsize=14, wrap=True)
            
            axes[0].imshow(transforms.ToPILImage()(all_images[img_idx].cpu().float()))
            axes[0].set_title("Original")
            axes[1].imshow(transforms.ToPILImage()(all_blurryrecons[img_idx].cpu().float()))
            axes[1].set_title("Blurry")
            axes[2].imshow(transforms.ToPILImage()(all_recons[img_idx].cpu().float()))
            axes[2].set_title("Recon")
            axes[3].imshow(final_pil_img)
            axes[3].set_title("Enhanced")
            
            for ax in axes: ax.axis('off')
            plt.tight_layout()
            plt.savefig(f"{png_dir}/combined_img_{img_idx}.png", bbox_inches='tight', dpi=150)
            plt.close() 


            samples = samples.cpu()[None]
            if all_enhancedrecons is None:
                all_enhancedrecons = samples
            else:
                all_enhancedrecons = torch.vstack((all_enhancedrecons, samples))

# Saving
all_enhancedrecons = transforms.Resize((256, 256))(all_enhancedrecons).float()
print("all_enhancedrecons", all_enhancedrecons.shape)
torch.save(all_enhancedrecons, f"evals/{model_name}/{model_name}_all_enhancedrecons.pt")
print(f"saved evals/{model_name}/{model_name}_all_enhancedrecons.pt")

if not utils.is_interactive():
    sys.exit(0)